
# 07. Каскадная маршрутизация: малая модель → эскалация на большую

Демо к модулю 3: сравнение стратегии `all_strong` (все задачи решает большая модель) и `cascade` (router отправляет простые задачи на дешёвую модель, сложные — на большую).

В ноутбуке используются mock-модели, чтобы пример запускался без API-ключей. В реальном проекте функцию `mock_model_call` нужно заменить на вызов OpenAI-совместимого API и оставить ту же схему логирования: качество, стоимость, latency, доля эскалаций, ошибки router.


In [1]:

from dataclasses import dataclass
from typing import Literal
import random
import pandas as pd


In [2]:

random.seed(42)

@dataclass
class Task:
    task_id: int
    text: str
    difficulty: Literal["easy", "medium", "hard"]
    risk: Literal["low", "medium", "high"]
    input_tokens: int
    expected_quality_threshold: float

tasks = []
for i in range(1, 31):
    if i <= 12:
        tasks.append(Task(i, f"Типовой запрос клиента #{i}", "easy", "low", random.randint(250, 600), 0.70))
    elif i <= 22:
        tasks.append(Task(i, f"Составной запрос с несколькими условиями #{i}", "medium", "medium", random.randint(600, 1200), 0.78))
    else:
        tasks.append(Task(i, f"Сложный рискованный запрос с политиками и исключениями #{i}", "hard", "high", random.randint(1200, 2200), 0.85))

pd.DataFrame([t.__dict__ for t in tasks]).head()


,task_id,text,difficulty,risk,input_tokens,expected_quality_threshold
0,1,Типовой запрос клиента #1,easy,low,577,0.7
1,2,Типовой запрос клиента #2,easy,low,307,0.7
2,3,Типовой запрос клиента #3,easy,low,262,0.7
3,4,Типовой запрос клиента #4,easy,low,390,0.7
4,5,Типовой запрос клиента #5,easy,low,375,0.7


In [3]:

model_prices = {
    "cheap": {"input_per_1k": 0.002, "output_per_1k": 0.006, "base_latency": 0.7},
    "strong": {"input_per_1k": 0.015, "output_per_1k": 0.060, "base_latency": 2.2},
}

def estimate_output_tokens(task: Task) -> int:
    return {"easy": 180, "medium": 300, "hard": 450}[task.difficulty]

def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    p = model_prices[model]
    return input_tokens / 1000 * p["input_per_1k"] + output_tokens / 1000 * p["output_per_1k"]

def mock_model_call(model: str, task: Task) -> dict:
    """Заглушка модели: качество и latency зависят от сложности задачи и класса модели."""
    output_tokens = estimate_output_tokens(task)
    base_quality = {
        ("cheap", "easy"): 0.82,
        ("cheap", "medium"): 0.70,
        ("cheap", "hard"): 0.52,
        ("strong", "easy"): 0.88,
        ("strong", "medium"): 0.86,
        ("strong", "hard"): 0.83,
    }[(model, task.difficulty)]
    quality = max(0, min(1, random.gauss(base_quality, 0.05)))
    latency = model_prices[model]["base_latency"] + 0.0015 * output_tokens + random.random() * 0.4
    cost = estimate_cost(model, task.input_tokens, output_tokens)
    return {"model": model, "quality": quality, "latency": latency, "cost": cost, "output_tokens": output_tokens}



## Router и критерии эскалации

Router может быть эвристическим, ML-классификатором или маленькой LLM 0.5–3B. В демо ниже используется простая эвристика.

Эскалация на большую модель происходит, если:

- сложность `hard`;
- риск `high`;
- запрос слишком длинный;
- cheap-модель дала качество ниже порога;
- нарушен формат / не хватает фактов / нужна проверка critic.


In [4]:

def router(task: Task) -> bool:
    """True = эскалировать на strong до генерации cheap-моделью."""
    if task.difficulty == "hard":
        return True
    if task.risk == "high":
        return True
    if task.input_tokens > 1500:
        return True
    return False

def run_all_strong(tasks):
    rows = []
    for task in tasks:
        result = mock_model_call("strong", task)
        rows.append({"strategy": "all_strong", "task_id": task.task_id, "difficulty": task.difficulty,
                     "risk": task.risk, "escalated": True, **result})
    return pd.DataFrame(rows)

def run_cascade(tasks, post_check=True):
    rows = []
    for task in tasks:
        pre_escalate = router(task)
        if pre_escalate:
            result = mock_model_call("strong", task)
            escalated = True
            route_reason = "pre_router"
        else:
            cheap = mock_model_call("cheap", task)
            if post_check and cheap["quality"] < task.expected_quality_threshold:
                result = mock_model_call("strong", task)
                # учитываем, что cheap уже был вызван
                result["cost"] += cheap["cost"]
                result["latency"] += cheap["latency"]
                escalated = True
                route_reason = "post_quality_check"
            else:
                result = cheap
                escalated = False
                route_reason = "cheap_ok"
        rows.append({"strategy": "cascade", "task_id": task.task_id, "difficulty": task.difficulty,
                     "risk": task.risk, "escalated": escalated, "route_reason": route_reason, **result})
    return pd.DataFrame(rows)

df = pd.concat([run_all_strong(tasks), run_cascade(tasks)], ignore_index=True)
df.head()


,strategy,task_id,difficulty,risk,escalated,model,quality,latency,cost,output_tokens,route_reason
0,all_strong,1,easy,low,True,strong,0.862227,2.773523,0.019455,180,NaN
1,all_strong,2,easy,low,True,strong,0.979289,2.533864,0.015405,180,NaN
2,all_strong,3,easy,low,True,strong,0.844330,2.556126,0.014730,180,NaN
3,all_strong,4,easy,low,True,strong,0.898853,2.775398,0.016650,180,NaN
4,all_strong,5,easy,low,True,strong,0.919145,2.613592,0.016425,180,NaN


In [5]:

summary = (
    df.groupby("strategy")
      .agg(
          mean_quality=("quality", "mean"),
          total_cost=("cost", "sum"),
          mean_latency=("latency", "mean"),
          p95_latency=("latency", lambda x: x.quantile(0.95)),
          escalation_rate=("escalated", "mean"),
      )
      .reset_index()
)
summary


,strategy,mean_quality,total_cost,mean_latency,p95_latency,escalation_rate
0,all_strong,0.874114,0.934050,2.855921,3.190407,1.000000
1,cascade,0.838459,0.758285,2.650446,4.433808,0.566667


In [6]:

# Разбор ошибок router: где cascade оставил cheap-модель, но качество ниже требуемого порога.
cascade = df[df["strategy"] == "cascade"].copy()
task_thresholds = {t.task_id: t.expected_quality_threshold for t in tasks}
cascade["threshold"] = cascade["task_id"].map(task_thresholds)
cascade["failed_quality"] = cascade["quality"] < cascade["threshold"]

router_failures = cascade[(cascade["escalated"] == False) & (cascade["failed_quality"])]
router_failures[["task_id", "difficulty", "risk", "model", "quality", "threshold", "cost", "latency"]]


,task_id,difficulty,risk,model,quality,threshold,cost,latency



## Что написать в отчёте

1. Сравнить `all_strong` и `cascade` по стоимости, качеству, latency и доле эскалаций.
2. Показать, какие задачи router отправил на дешёвую модель, а какие — на большую.
3. Разобрать ошибки router.
4. Объяснить, когда каскад выгоден:
   - много простых задач;
   - router редко ошибается;
   - большая модель существенно дороже;
   - требования к latency допускают двухшаговую обработку.
5. Объяснить, когда каскад опасен:
   - высокая цена ошибки;
   - плохо формализованы критерии эскалации;
   - cheap-модель уверенно ошибается;
   - нет post-check / critic.
